# Car Loan EMI Repayment — Monthly Status Pipeline

## What this notebook does
This notebook manages the car loan EMI repayment schedule in `personal.mohan.repayment_schedule`, built from `Repayment_Schedule.xlsx` (stored in `/Volumes/personal/mohan/files`). The table holds **84 monthly instalments** for a 950,000 car loan, from 18-Oct-2026 to 18-Aug-2033 — bank instalment 14,688.86 per month, total interest 283,863.83.

**Cell sequence (run top to bottom):**
1. **Column renames** — renames `actual_instalment` to `paid_installment` and `bank_instalment` to `bank_installment` (guarded, so monthly re-runs are safe).
2. **updated_timestamp setup** — one-time setup: enables column defaults and gives `updated_timestamp` a `DEFAULT current_timestamp()`; already-paid instalments are stamped with their payment date. Unpaid rows keep NULL until they are actually marked paid.
3. **Monthly auto-update (pipeline core)** — marks every instalment that is due (`dates <= today`) and not yet paid as `status = 'paid'`, stamping `updated_timestamp` at the exact moment of the update. It is idempotent (re-running changes nothing) and catches up automatically if a monthly run is ever missed.
4. **Validation summary** — reports which month payments are paid through, how many instalments are paid and pending, and the remaining outstanding principal. After each monthly run this shows the up-to-date repayment status.

## The scheduled pipeline (job: `car_montly_emi_status`)
* **Schedule:** 18th of every month at 9:00 PM IST, first run 18-Dec-2026. A saved gate query (`Repayment auto-update start gate (from 2026-12-18)`) blocks any run before that date.
* **Each run:** executes this notebook top to bottom, marking the instalment(s) that came due that month as paid.
* **Notification:** an email is sent to mohangodm@gmail.com after every run — on success and on failure.

In [0]:
%sql
-- updated_timestamp column was added with ALTER TABLE ... ADD COLUMN updated_timestamp TIMESTAMP;
-- Delta requires enabling column defaults on the table before setting a DEFAULT
ALTER TABLE personal.mohan.repayment_schedule
SET TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE personal.mohan.repayment_schedule
ALTER COLUMN updated_timestamp SET DEFAULT current_timestamp();

-- updated_timestamp = when the status was actually updated:
-- already-paid rows get their payment date (from `dates`); unpaid rows stay NULL
-- until the monthly job marks them paid
UPDATE personal.mohan.repayment_schedule
SET updated_timestamp = CASE WHEN status = 'paid' THEN CAST(dates AS TIMESTAMP) END;

In [0]:
%sql
-- Monthly auto-update: mark all due instalments (dates <= today) as paid
-- Scheduled: 18th of every month at 9:00 PM IST starting 2026-12-18
UPDATE personal.mohan.repayment_schedule
SET status = 'paid',
    updated_timestamp = current_timestamp()
WHERE dates <= current_date()
  AND (status IS NULL OR status <> 'paid')

In [0]:
%sql
-- Validation summary (vertical) — runs last, after the monthly auto-update:
-- which month payments are paid through, how many instalments are paid vs pending,
-- and the remaining outstanding principal
WITH summary AS (
  SELECT
    MAX(CASE WHEN status = 'paid' THEN dates END) AS paid_through_month,
    MIN(CASE WHEN status IS NULL OR status <> 'paid' THEN dates END) AS next_due_month,
    COUNT(CASE WHEN status = 'paid' THEN 1 END) AS instalments_paid,
    COUNT(CASE WHEN status IS NULL OR status <> 'paid' THEN 1 END) AS instalments_pending,
    (SELECT CAST(outstanding_principal AS DECIMAL(18,2)) FROM personal.mohan.repayment_schedule
      WHERE status IS NULL OR status <> 'paid'
      ORDER BY instalment_no LIMIT 1) AS remaining_outstanding_principal
  FROM personal.mohan.repayment_schedule
)
SELECT stack(5,
  'Last paid month',              CAST(paid_through_month AS STRING),
  'Next due month',                  CAST(next_due_month AS STRING),
  'Instalments paid',                CAST(instalments_paid AS STRING),
  'Instalments pending',             CAST(instalments_pending AS STRING),
  'Remaining outstanding principal', CAST(remaining_outstanding_principal AS STRING)
) AS (metric, value)
FROM summary